In [4]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
from pathlib import Path
from evaluation import mean_per_user_f1, search_threshold
from models import load_model_artifacts
import lightgbm as lgb
from calibration import evaluate_calibration_cv

# Instacart Market Basket Analysis

## Phase 4: Evaluation

This notebook is self-contained. It reloads the feature matrix and the three trained models saved in Phase 3, then runs the evaluation analyses: probability calibration, decision threshold tuning and the $\tau^* = F_1^*/2$ check, per-user versus global threshold gap, feature ablation, the corrected model comparison test, and finally a single sealed test-set evaluation. Loading models from disk rather than depending on the modeling notebook's memory means this notebook runs top to bottom on its own.

In [2]:
PROCESSED_DIR = Path("../data/processed")
matrix = pd.read_parquet(PROCESSED_DIR / "feature_matrix.parquet")
train = matrix[matrix["split"] == "train"].copy()
test = matrix[matrix["split"] == "test"].copy()

print(f"Train: {train.shape}, users: {train['user_id'].nunique():,}")
print(f"Test:  {test.shape}, users: {test['user_id'].nunique():,}")

Train: (6359442, 20), users: 98,406
Test:  (2115219, 20), users: 32,803


In [6]:
logistic = load_model_artifacts("../models/logistic_v1", "sklearn")
lgbm = load_model_artifacts("../models/lgbm_v1", "lightgbm")

feature_cols = lgbm["feature_cols"]  

print("Loaded models:")
print(f"  logistic: threshold {logistic['threshold']:.3f}, CV F1 {logistic['cv_scores']['mean_f1']:.4f}")
print(f"  lgbm:     threshold {lgbm['threshold']:.3f}, CV F1 {lgbm['cv_scores']['mean_f1']:.4f}")
print(f"  features ({len(feature_cols)}): {feature_cols}")

Loaded models:
  logistic: threshold 0.162, CV F1 0.3593
  lgbm:     threshold 0.192, CV F1 0.3734
  features (16): ['count_up', 'f_up', 'recency_up', 'mean_cart_position_up', 'N_u', 'mean_basket_size_u', 'mean_interval_u', 'var_interval_u', 'interval_censored_u', 'reorder_ratio_u', 'unique_products_u', 'r_p_shrunk', 'unique_buyers_p', 'mean_cart_position_p', 'r_aisle', 'r_dept']


### Probability Calibration: Isotonic Regression

Threshold tuning and the $\tau^* = F_1^*/2$ result both assume the model's scores are genuine probabilities, not merely correctly-ordered ranking scores. A gradient-boosted model can rank pairs perfectly while being systematically over- or under-confident: it may output 0.3 for a group of pairs that actually reorder 45% of the time. The ranking is correct, the numbers are not. Calibration corrects the numbers without disturbing the ranking.

#### What calibration must satisfy

A classifier is calibrated when its output equals the true conditional frequency:

$$P\big(y = 1 \mid \hat{s}(x) = p\big) = p \quad \text{for all } p,$$

that is, among all pairs scored $p$, a fraction $p$ actually reorder. Isotonic regression learns a correction $g$ mapping raw scores to calibrated probabilities under a single constraint: $g$ must be monotonic non-decreasing. Monotonicity encodes the decision to trust the model's ranking and adjust only the magnitudes, if raw score $a > b$, then $g(a) \geq g(b)$.

#### The optimisation problem

Given calibration-set raw scores $s_1 \leq s_2 \leq \dots \leq s_n$ (sorted) with outcomes $y_1, \dots, y_n \in \{0, 1\}$, isotonic regression solves

$$\min_{g_1 \leq g_2 \leq \dots \leq g_n} \sum_{i=1}^{n} (g_i - y_i)^2,$$

the best least-squares fit to the outcomes subject to the values never decreasing. The solution is a non-decreasing step function.

#### The algorithm: Pool Adjacent Violators (PAVA)

The constrained minimiser is found exactly, not approximately, by Pool Adjacent Violators. Initialise each fitted value at its raw outcome, $g_i = y_i$. Scan for any adjacent pair that decreases ($g_i > g_{i+1}$, a *violation* of monotonicity) and replace both by their weighted average (*pooling*). Pooling repeats, merging blocks until the whole sequence is non-decreasing. Each final pooled block takes the value

$$g_{\text{block}} = \frac{1}{|B|} \sum_{i \in B} y_i,$$

which is exactly the observed reorder frequency within that block. This is why the output is calibrated by construction: every fitted value is a measured frequency of positives.

#### Worked example

Sorted outcomes $[0, 0, 1, 0, 1, 1]$. The subsequence $1, 0$ (positions 3 to 4) decreases, a violation. Pool into their average $0.5$:

$$[0,\ 0,\ 1,\ 0,\ 1,\ 1] \;\longrightarrow\; [0,\ 0,\ 0.5,\ 0.5,\ 1,\ 1].$$

The sequence is now non-decreasing, so PAVA terminates. The learned map sends the lowest scores to $0$, the pooled middle to $0.5$, and the top to $1$. The middle block maps to $0.5$ because exactly half of the outcomes that landed there were positive, an honest probability by definition.

#### Applying the correction

For a new prediction, the raw score is located on this staircase and returned as its block value. Because the staircase never steps down, ranking is preserved exactly; because each step equals an observed frequency, the returned values are calibrated. The correction fixes magnitudes while leaving the order, and therefore the model's discriminative quality (AUC), untouched.

#### Leakage discipline

The staircase is fit on a calibration slice disjoint from the validation fold. Fitting and evaluating on the same data would make it perfect by construction, since the step values are the outcomes it was fit to, so the routine here fits the calibrator on one held-out group of users and measures its effect (Brier decomposition and mean per-user F1) on a separate untouched fold.

In [5]:
best_params = lgbm["cv_scores"]["extra"]["best_params"]

cal_model = lgb.LGBMClassifier(
    n_estimators=1000,
    random_state=42,
    verbosity=-1,
    **best_params,
)

cal_results = evaluate_calibration_cv(
    estimator=cal_model,
    X=train[feature_cols],
    y=train["y"].values,
    groups=train["user_id"].values,
    n_splits=5,
    n_bins=10,
)

print("Brier score (raw vs calibrated), per fold:")
for i, (r, c) in enumerate(zip(cal_results["raw_brier"], cal_results["cal_brier"])):
    print(f"  fold {i}: raw {r['brier']:.5f} -> cal {c['brier']:.5f}")

raw_rel = np.mean([b["reliability"] for b in cal_results["raw_brier"]])
cal_rel = np.mean([b["reliability"] for b in cal_results["cal_brier"]])
print(f"\nReliability term (calibration error), mean: raw {raw_rel:.6f} -> cal {cal_rel:.6f}")
print(f"Mean per-user F1: raw {cal_results['mean_raw_f1']:.4f} -> calibrated {cal_results['mean_cal_f1']:.4f}")

Brier score (raw vs calibrated), per fold:
  fold 0: raw 0.07214 -> cal 0.07215
  fold 1: raw 0.07080 -> cal 0.07081
  fold 2: raw 0.07124 -> cal 0.07126
  fold 3: raw 0.07154 -> cal 0.07155
  fold 4: raw 0.07132 -> cal 0.07133

Reliability term (calibration error), mean: raw 0.000002 -> cal 0.000004
Mean per-user F1: raw 0.3733 -> calibrated 0.3734


The raw per-user F1 in this calibration analysis (0.3733) is a hair below the modeling notebook's 0.3734 because the calibration procedure reserves 25% of each fold's training users to fit the isotonic calibrator, so the model here is trained on less data by construction. The difference (0.0001) is the cost of that reserved slice, not a change in model quality. The calibrated score returning to 0.3734 is within threshold-search noise, consistent with the reliability term showing the model was already calibrated and isotonic therefore having nothing to correct. The headline F1 remains 0.3734 from the full-data model.

#### Findings: Calibration

The raw reliability term (calibration error) is 0.000002, essentially zero: the LightGBM model is already almost perfectly calibrated before any correction. Isotonic regression therefore has nothing to fix, the Brier score is unchanged to four decimal places, and mean per-user F1 moves by 0.0001 (0.3733 to 0.3734), within noise. The calibrated reliability term is marginally higher (0.000004), reflecting isotonic regression fitting slight staircase noise to a relationship that was already straight.

This near-perfect calibration is expected: LightGBM optimises log-loss by default, which rewards probability quality directly rather than ranking alone, so a well-tuned gradient-boosted model often emerges calibrated without post-hoc correction.

The finding matters for two later results. First, it validates the decision threshold: $\tau^*$ was tuned on the raw probabilities assuming they were genuine, and this confirms they were, so the threshold sits on a true probability axis. Second, it licenses the $\tau^* = F_1^*/2$ theorem, which requires calibration as a precondition. LightGBM's observed threshold (0.192) sits almost exactly at $F_1^*/2 = 0.187$; because the model is now shown to be calibrated, this agreement is the theorem holding, not a coincidence. The contrast with the frequency floor is explanatory: the floor's raw $f_{u,p}$ is not a calibrated probability, its threshold (0.26) sits far from its own $F_1^*/2$ (0.164), and the theorem correctly fails to apply there while holding for the calibrated LightGBM.